# 02 — Grouped Split, Baseline Benchmarking & Model Training

**Project:** FlyRank Ranking Signal Analysis — Predicting Continuous Search Visibility  
**Pipeline Stage:** Notebook 02 of 03 (Model Training & Benchmark Optimization)  
**Target Variable:** Continuous Search Visibility $\log(1 + \text{impressions\_90d})$ on Held-Out Enterprise Client Portfolios  
**Standardized Split:** Grouped Holdout (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`) with Zero Client Domain Overlap  

---

### Notebook Architecture & Objectives
This notebook is your starting template for model training and optimization:
1. **Grouped Holdout Split Design:** Enforce a strict zero-client-leakage partition on `client_id` with programmatic disjointness assertions.
2. **Transparent Heuristic Baselines:** Fit Baseline 1 (Mean Dummy Regressor) and Baseline 2 (Univariate Word Count Rule) to establish the floor to beat.
3. **High-Accuracy Model Training:** 
   - Fit candidate algorithms (e.g., cross-validated Polynomial Interaction Ridge, HistGradientBoosting Regressor, or Stacking Ensemble).
   - Maximize generalization accuracy on unseen client domains.
4. **Payload Export:** Export model predictions, test set ground truth, and metadata to `work/outputs/model_evaluation_payload.pkl` for evaluation in Notebook 03.

## 1. Grouped Holdout Split Design (Zero Client Leakage)

### 1.1 The Multi-Tenant Generalization Challenge
In multi-tenant SaaS platforms, standard random row splits produce severe **client leakage**: content items from the same client share identical domain authority, root backlink velocity, CMS infrastructure, and brand equity. A random split allows models to memorize client-level baseline volume, inflating test performance while collapsing on brand-new client domains.

### 1.2 The Client Holdout Standard
To guarantee 100% scientific validity:
* We partition strictly by `client_id` using `GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)`.
* We assert that training clients and test clients are mutually exclusive:
  $$\text{set}(\text{clients}_{\text{train}}) \cap \text{set}(\text{clients}_{\text{test}}) = \emptyset$$
* Test metrics reflect true generalization performance on entirely unseen enterprise domains.

In [ ]:
# Code Cell 1: Load Features and Execute Grouped Split
# TODO:
# 1. Load prepared feature matrix from 'work/outputs/prepared_features.parquet' (or .csv).
# 2. Separate feature matrix X, target y (target_log_impressions), and groups (client_id).
# 3. Perform GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42).
# 4. Programmatically assert set(clients_train).isdisjoint(set(clients_test)).
# 5. Print train and test matrix shapes and target distributions.


## 2. Transparent Heuristic Baseline Benchmarks

To verify that machine learning provides genuine predictive lift rather than fitting noise, evaluate two non-ML baselines on the exact same test cohort:
1. **Baseline 1 (Mean Dummy Regressor):** Always predicts the training set target mean $\bar{y}_{\text{train}}$. (Yields $R^2 = 0.0$ on train; negative $R^2$ on held-out clients with a lower mean).
2. **Baseline 2 (Univariate Word Count Rule):** A simple single-variable OLS regression on `log_word_count`, reflecting the common SEO belief that "longer articles rank higher and gain more visibility".

In [ ]:
# Code Cell 2: Fit and Evaluate Standardized Heuristic Baselines
# TODO:
# 1. Define an evaluation helper function computing R2 score, RMSE, MAE, and Spearman rho.
# 2. Fit Baseline 1: Mean Dummy Regressor (predicts y_train.mean()).
# 3. Fit Baseline 2: Univariate OLS Regressor using only 'log_word_count'.
# 4. Evaluate both baselines on the held-out test cohort (X_test, y_test).
# 5. Display the baseline performance summary table.


## 3. High-Accuracy Model Training

### 3.1 Model Architecture Strategy
To achieve the highest honest accuracy without data leakage, consider:
1. **Regularized Linear Models with Interactions:** Pairwise interaction terms ($x_i \times x_j$) across primary signals regularized with `RidgeCV`.
2. **Gradient Boosted Decision Trees (`HistGradientBoostingRegressor`):** Captures sharp non-linear thresholds, step-function jumps, and missingness natively.
3. **Champion Blended Ensemble:** Blending linear macro-trends with localized tree splits ($w \cdot \hat{y}_{\text{Ridge}} + (1-w) \cdot \hat{y}_{\text{HGBR}}$) to maximize variance explained.

In [ ]:
# Code Cell 3: Machine Learning Model Training & Optimization
# TODO:
# 1. Define your candidate model pipelines:
#    - Candidate A: Polynomial Interaction Ridge Regression with RidgeCV
#    - Candidate B: HistGradientBoostingRegressor (tuned learning_rate, max_iter, l2_regularization)
#    - Champion Architecture: Blended Stacking Ensemble (0.50 * Ridge + 0.50 * HGBR)
# 2. Fit model(s) on the training set (X_train, y_train).
# 3. Generate predictions on the unseen test set (X_test).
# 4. Compute test metrics (R2, RMSE, MAE, Spearman rho) and print comparison logs.


In [ ]:
# Code Cell 4: Export Model Evaluation Payload
# TODO:
# 1. Package test ground truth, predictions from all models, client IDs, and metadata into a dictionary payload.
# 2. Save payload to 'work/outputs/model_evaluation_payload.pkl' for Notebook 03.
# 3. Export predictions dataframe to 'work/outputs/model_test_predictions.csv'.
# 4. Print confirmation of export paths and file sizes.
